# Projeto Interconexão - ETL dos dados

A documentação do projeto pode ser encontrada em [Modelo_Interconexão_BRAIN_Guilherme_Andrade](https://algarnet.sharepoint.com/sites/TriboAnalytics246/Documentos%20Compartilhados/Forms/AllItems.aspx?id=%2Fsites%2FTriboAnalytics246%2FDocumentos%20Compartilhados%2FGeneral%2FSquad%20Data%20Wizards%20%2D%20Projetos%2FModelo%5FInterconex%C3%A3o%5FBRAIN%5FGuilherme%5FAndrade&viewid=18eaba8b%2D6b2b%2D40e3%2D85d6%2De3c0e7598b0c&FolderCTID=0x012000E7FCBA090D95C348A4A8E9F05538F026)

Nesse *pipeline* serão realizadas as seguintes ações:

1. Criação das funções auxiliares (ETL e Engenharia de *Features*),
2. Aplicação das funções de ETL para gerar a tabela processada,
3. Aplicação das funções de Engenharia de *Features*  
3.1. Geração da tabela **com** intervalo de tempo de definido  
3.2. Geração da tabela **sem** intervalo de tempo de definido  
4. Salvamento das tabelas

In [ ]:
# === Libraries
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, regexp_extract, lit, split, when
from snowflake.snowpark.types import TimestampType
from snowflake.snowpark.window import Window
from datetime import datetime, timedelta
from snowflake.snowpark.functions import udf
import snowflake.snowpark.functions as F

# === Settings
warnings.filterwarnings("ignore")
session = get_active_session()

### Funções Auxiliares

In [ ]:
# === Movendo o prefixo da coluna (col_name) para o final do nome: 1_Coluna para Coluna_1
def move_prefix_to_end(df):
    new_columns = {}
    for col_name in df.schema.names:
        clean_name = col_name.strip('"')
        if "_" in clean_name:
            parts = clean_name.split("_", 1)
            if parts[0].isdigit():
                col_body = parts[1].split()[0]
                new_name = f"{col_body}_{parts[0]}"
                new_columns[col_name] = new_name
    return df.select([df[col].alias(new_columns.get(col, col)) for col in df.schema.names])

In [ ]:
# === Validação para verificar consistência dos dados de chamadas
def check_columns(df):
    invalid_indices_200 = df.filter((F.col('SipStatusCode_135') == 200) & (F.col('AccountingSessionTime_13') <= 0)).count()
    invalid_indices_not_200 = df.filter((F.col('SipStatusCode_135') != 200) & (F.col('AccountingSessionTime_13') == 0)).count()
    invalid_indices = invalid_indices_200 + invalid_indices_not_200

    if invalid_indices == 0:
        print("Sucesso")
    else:
        print(f"{invalid_indices} índices não atendem a regra de duração de chamadas")

In [ ]:
# === Remoção das colunas a não serem utilizadas
def id_cols(df):
    cols = ['ACCOUNTINGSESSIONID_4', 'INGRESSSESSIONID_5', 'EGRESSSESSIONID_6', 'FLOWIDENTIFIER_24', 'CISCODISCONNECTCAUSE_17',
            'FLOWIDENTIFIER_48', 'EGRESSVLANTAGVALUE_19', 'INGRESSVLANTAGVALUE_21', 'INGRESSREALM_23',
            'PRIMARYROUTINGNUMBER_121', 'FLOWINPUTREALM_26', 'FLOWINPUTSRCADDR_27', 'FLOWINPUTDESTADDRESS_29',
            'FLOWOUTPUTSRCADDRESS_32', 'FLOWOUTPUTDESTADDR_34', 'FLOWINPUTSRCADDR_51', 'FLOWINPUTDESTADDRESS_53',
            'FLOWOUTPUTREALM_55', 'FLOWOUTPUTSRCADDRESS_56', 'FLOWOUTPUTDESTADDR_58', 'INGRESSLOCALADDRESS_127',
            'INGRESSREMOTEADDRESS_128', 'EGRESSLOCALADDRESS_129', 'EGRESSREMOTEADDRESS_130', 'FLOWINPUTSRCPORT_52',
            'FLOWINPUTDESTPORT_54', 'FLOWOUTPUTSRCPORT_57', 'FLOWOUTPUTDESTPORT_59', 'FLOWINPUTSRCPORT_28',
            'FLOWINPUTDESTPORT_30', 'FLOWOUTPUTSRCPORT_33', 'FLOWOUTPUTDESTPORT_35', 'EGRESSROUTINGNUMBER_136',
            'RTCPCALLINGPACKETSLOST_36', 'RTCPCALLINGAVGJITTERMSEC_37', 'RTCPCALLINGMAXJITTERMSEC_39',
            'RTPCALLINGPACKETSLOST_41', 'RTPCALLINGAVGJITTERMSEC_42', 'RTPCALLINGMAXJITTERMSEC_43', 'RTPCALLINGOCTETS_44',
            'RTPCALLINGPACKETS_45', 'RTCPCALLEDPACKETSLOST_60', 'RTCPCALLEDAVGJITTERMSEC_61', 'RTCPCALLEDMAXJITTERMSEC_63',
            'RTPCALLEDPACKETSLOST_65', 'RTPCALLEDAVGJITTERMSEC_66', 'RTPCALLEDMAXJITTERMSEC_67', 'RTPCALLEDOCTETS_68',
            'RTPCALLEDPACKETS_69', 'CALLINGRFACTOR_46', 'CALLINGMOS_47', 'CALLEDRFACTOR_70', 'CALLEDMOS_71',
            'POSTDIALDELAYMSEC_120', 'DISCONNECTCAUSE_134', 'TERMINATINGTRUNKGROUP_123', 'PASSERTEDID_126'
]
    
    df = df.drop(*cols)
    
    return df

In [ ]:
# === Converte String para datetime
@udf(session=session, return_type=TimestampType())
def transform_datetime(input_str: str) -> datetime: # === Recebe uma string

    if not input_str:
        return None

    try:    
        # === Remover milissegundos e fuso horário
        time_str = input_str.split('.')[0].split(' ')[0]

        # === Extrair parte da data
        date_part = input_str.split(' ')[-3:]
        date_str = ' '.join(date_part)

        # === Combinar partes de data e hora em uma única string
        datetime_str = f"{date_str} {time_str}"

        # === Converter para objeto datetime
        return datetime.strptime(datetime_str, "%b %d %Y %H:%M:%S")
    except Exception:
        return None

In [ ]:
def process(df):
    # === Remover duplicatas
    df = df.drop_duplicates()
    
    # === Remover colunas de identificação baseado na função da célula remove_cols
    df = id_cols(df)
    print("Duplicatas e colunas de identificação retiradas!")
    
    # === Verificar regra de consistência
    cond_1970 = col('CISCOCONNECTTIME_15') == '00:00:00.000 UTC Jan 01 1970'
    cond_zero = col('ACCOUNTINGSESSIONTIME_13') == 0
    
    verificacao = df.filter(cond_1970 & cond_zero).count() == df.filter(cond_1970).count()
    
    if verificacao:
        print("Todas as linhas onde a coluna 15 é '1970' têm a coluna 13 igual a 0.")
    else:
        print("Existem linhas onde a coluna 15 é '1970' e a coluna 13 não é igual a 0.")
        
    # === Aplicar transformação de String e em datetime
    df = df.with_column('datahora', transform_datetime(col('CISCOSETUPTIME_14')))
    df = df.drop('CISCOSETUPTIME_14')
    print("Datahora criada!")
    
    # === Criar colunas de identificação a partir de SIP
    df = df.with_column('IDENTIFICATIONNUMBER_A_10', regexp_extract(col('CALLINGSTATIONID_10'), r'sip:(\d+)@', 1))
    df = df.with_column('IDENTIFICATIONNUMBER_B_11', regexp_extract(col('CALLEDSTATIONID_11'), r'sip:(\d+)@', 1))
    df = df.with_column('NUMBER_A_IP_10', split(regexp_extract(col('CALLINGSTATIONID_10'), r'@([^>]+)>', 1), lit(';'))[0])
    df = df.with_column('NUMBER_B_IP_11', split(regexp_extract(col('CALLEDSTATIONID_11'), r'@([^>]+)>', 1), lit(';'))[0])
    
    df = df.drop('CALLEDSTATIONID_11', 'CALLINGSTATIONID_10')

    
    # === Preencher valores nulos com 0
    df = df.fillna(0)
    return df

In [ ]:
# === Agrupamentos pela origem e conta os tipos de erro
def agrupamentos_por_origem(df, X):
    
    # === Assegura timestamp e ordenação
    if dict(df.dtypes).get("DATAHORA") != "timestamp":
        df = df.withColumn("DATAHORA", F.to_timestamp("DATAHORA"))
        print('Coluna DATAHORA convertida.')

    # === Define o limite
    min_time = df.agg(F.min("DATAHORA")).first()[0]
    threshold_time = min_time + timedelta(minutes=X)
    df = df.filter(F.col("DATAHORA") >= threshold_time)
    print('Filtro de DATAHORA aplicado.')

    # === Define janela de X minutos para cada origem
    window_spec = (
        Window
        .partitionBy("IDENTIFICATIONNUMBER_B_11")
        .orderBy(F.unix_timestamp("DATAHORA"))
        .rangeBetween(-X * 60, 0)
    )

    # === Todos os tipos de erro distintos
    #tipos_de_erros = ['483', '433', '400', '484', '407', '420', '422', '0', '452', '604', '580', '503', '486',
    #                  '200', '488', '487', '404', '405', '501', '411', '500', '414', '482', '606', '481', '600',
    #                  '491', '504', '302', '406', '408', '415', '502', '410', '401', '403', '570', '421', '453',
    #                  '480', '436', '603', '476']

    # === Erros selecionados
    tipos_de_erros = ['200', '404', '487']

    # === Conta a quantidade de tipos de erros e cria sua respectiva coluna.
    novas_colunas = {
        f"erro_{erro}_{X}min": F.sum(
            F.when(F.col("SIPSTATUSCODE_135") == erro, 1).otherwise(0)
        ).over(window_spec)
        for erro in tipos_de_erros
    }
 
    novas_colunas.update({
        f"duracao_media_{X}min": F.avg("ACCOUNTINGSESSIONTIME_13").over(window_spec),
        f"origens_diferentes_{X}min": F.size(
            F.array_distinct(F.collect_list("IDENTIFICATIONNUMBER_A_10").over(window_spec))
        ),
        f"portas_diferentes_{X}min": F.size(
            F.array_distinct(F.collect_list("NUMBER_A_IP_10").over(window_spec))
        ),
    })

    # === Adiciona colunas ao dataframe
    for nome_coluna, expressao in novas_colunas.items():
        df = df.with_column(nome_coluna, expressao)

    # === Remove colunas e tempo_chamada NA com 0.
    colunas_a_remover = [col for col in ['CISCOCONNECTTIME_15', 'CISCODISCONNECTTIME_16'] if col in df.columns]
    df = df.drop(*colunas_a_remover)
    print(f'Colunas removidas: {colunas_a_remover}.')
 
    colunas_numericas = [nome for nome, tipo in df.dtypes if tipo in ('int', 'bigint', 'float', 'double')]
    df = df.fillna(0, subset=colunas_numericas)
    print(f'Preenchimento de colunas realizado: {colunas_numericas}.')
 
    return df

In [ ]:
def tempo_chamada(df):
    # === Contador de registros alterados
    registros_alterados = df.filter((col("ACCOUNTINGSESSIONTIME_13") < 3) & (col("SIPSTATUSCODE_135") == 200)).count()
    
    # === Atualiza a coluna 'target'
    df = df.with_column(
        "SIPSTATUSCODE_135",
        when((col("ACCOUNTINGSESSIONTIME_13") < 3) & (col("SIPSTATUSCODE_135") == 200), 999).otherwise(col("SIPSTATUSCODE_135"))
    ) 
    
    # === Printar quantos casos ocorreram
    print(f"{registros_alterados} registros foram alterados por conta da duração da chamada, tendo seu status atualizado para falha local")
    
    return df

In [ ]:
df = session.table("DWDEV.DATASCIENCE.F_VOICETRAFFIC_SBC_RAW")
print(f"Máximo de observações: {df.count()}")
df.show()

In [ ]:
df_renomeado = move_prefix_to_end(df)
df_renomeado.show()

In [ ]:
df_renomeado = df_renomeado.with_column(
        "SIPSTATUSCODE_135",
        when((col("ACCOUNTINGSESSIONTIME_13") < 3) & (col("SIPSTATUSCODE_135") == 200), 999).otherwise(col("SIPSTATUSCODE_135"))
    )

In [ ]:
# === Aplicar as funções
df_process = process(df_renomeado)
df_process.show()

In [ ]:
df_process = tempo_chamada(df_process)
df_process.show()

In [ ]:
# === Salvar df_process no Snowflake
df_process.write.mode("overwrite").save_as_table("F_VOICETRAFFIC_SBC_PROCESSED")

In [ ]:
df_processed = session.table("DWDEV.DATASCIENCE.F_VOICETRAFFIC_SBC_PROCESSED")

In [ ]:
df_processed = df_processed.filter(col("SIPSTATUSCODE_135").isin([200,
                                                                  404,
                                                                  #487
                                                                 ]))
df_processed.group_by("SIPSTATUSCODE_135").count().show()

In [ ]:
df_processed = df_processed.select(
    col("SIPSTATUSCODE_135"),
    col("NUMBER_A_IP_10"), 
    col("NUMBER_B_IP_11"),
    col("DATAHORA"),
    col("ACCOUNTINGSESSIONTIME_13"),
    col("IDENTIFICATIONNUMBER_A_10"),
    col("IDENTIFICATIONNUMBER_B_11")
)
df_processed

In [ ]:
# === Agrupamentos pela origem em janelas fixas de tempo configurável
def agrupamentos_por_origem_janela_fixa(df, horas=24):
    """
    Agrupa dados por origem em janelas fixas de tempo.
    
    Args:
        df: DataFrame com os dados
        horas: Número de horas para a janela (padrão: 24h)
    
    Returns:
        DataFrame agregado por origem e janela de tempo
    """
    
    # === Assegura timestamp e ordenação
    if dict(df.dtypes).get("DATAHORA") != "timestamp":
        df = df.withColumn("DATAHORA", F.to_timestamp("DATAHORA"))
        print('Coluna DATAHORA convertida.')
    
    # === Cria janelas fixas baseadas no timestamp
    # Calcula o número de segundos na janela
    segundos_janela = horas * 3600
    
    # === Encontra o timestamp mínimo para usar como referência
    min_timestamp = df.agg(F.min("DATAHORA")).collect()[0][0]
    
    # Cria ID da janela baseado no tempo decorrido desde o início
    df = df.withColumn(
        "JANELA_ID", 
        F.floor(
            (F.unix_timestamp("DATAHORA") - F.unix_timestamp(F.lit(min_timestamp))) / segundos_janela
        ).cast("int")
    )
    
    # Cria timestamp de início da janela para referência
    df = df.withColumn(
        "INICIO_JANELA",
        F.from_unixtime(
            F.unix_timestamp(F.lit(min_timestamp)) + F.col("JANELA_ID") * segundos_janela
        ).cast("timestamp")
    )
    
    # === Erros selecionados
    tipos_de_erros = ['200', '404', '487']
    
    # === Agregações por origem e por janela de tempo
    df_agregado = df.groupBy("IDENTIFICATIONNUMBER_B_11", "JANELA_ID", "INICIO_JANELA").agg(
        # Contadores de erro por janela
        *[F.sum(F.when(F.col("SIPSTATUSCODE_135") == erro, 1).otherwise(0)).alias(f"erro_{erro}_{horas}h") 
          for erro in tipos_de_erros],
        
        # Métricas de comportamento por janela
        F.avg("ACCOUNTINGSESSIONTIME_13").alias(f"duracao_media_{horas}h"),
        F.countDistinct("IDENTIFICATIONNUMBER_A_10").alias(f"origens_diferentes_{horas}h"),
        F.countDistinct("NUMBER_A_IP_10").alias(f"portas_diferentes_{horas}h"),
        
        # Métricas adicionais
        F.count("*").alias(f"total_conexoes_{horas}h"),
        F.min("DATAHORA").alias("primeira_conexao_janela"),
        F.max("DATAHORA").alias("ultima_conexao_janela")
    )
    
    # === Adiciona timestamp de fim da janela para referência
    df_agregado = df_agregado.withColumn(
        "FIM_JANELA",
        F.from_unixtime(
            F.unix_timestamp("INICIO_JANELA") + segundos_janela
        ).cast("timestamp")
    )
    
    # === Remove colunas desnecessárias do DataFrame original se existirem
    colunas_a_remover = [col for col in ['CISCOCONNECTTIME_15', 'CISCODISCONNECTTIME_16'] 
                        if col in df.columns]
    if colunas_a_remover:
        df = df.drop(*colunas_a_remover)
        print(f'Colunas removidas: {colunas_a_remover}')
    
    # === Preenche valores nulos com 0 nas colunas numéricas
    colunas_numericas = [nome for nome, tipo in df_agregado.dtypes 
                        if tipo in ('int', 'bigint', 'float', 'double')]
    df_agregado = df_agregado.fillna(0, subset=colunas_numericas)
    print(f'Preenchimento realizado nas colunas: {colunas_numericas}')
    
    # === Ordena por origem e janela
    df_agregado = df_agregado.orderBy("IDENTIFICATIONNUMBER_B_11", "JANELA_ID")
    
    total_janelas = df_agregado.select("JANELA_ID").distinct().count()
    print(f'Agregação em janelas de {horas}h concluída. Total de janelas: {total_janelas}')
    
    return df_agregado


# === Função alternativa para manter dados originais + features agregadas
def adicionar_features_janela_fixa(df, horas=24):
    """
    Adiciona features de janela fixa ao DataFrame original (sem perder registros individuais)
    
    Args:
        df: DataFrame com os dados
        horas: Número de horas para a janela (padrão: 24h)
    
    Returns:
        DataFrame original com colunas adicionais das estatísticas da janela
    """
    
    # === Assegura timestamp
    if dict(df.dtypes).get("DATAHORA") != "timestamp":
        df = df.withColumn("DATAHORA", F.to_timestamp("DATAHORA"))
    
    # === Cria janelas fixas
    segundos_janela = horas * 3600
    min_timestamp = df.agg(F.min("DATAHORA")).collect()[0][0]
    
    df = df.withColumn(
        "JANELA_ID", 
        F.floor(
            (F.unix_timestamp("DATAHORA") - F.unix_timestamp(F.lit(min_timestamp))) / segundos_janela
        ).cast("int")
    )
    
    # === Cria window spec por origem e janela
    window_janela = Window.partitionBy("IDENTIFICATIONNUMBER_B_11", "JANELA_ID")
    
    tipos_de_erros = ['200',
                      '404',
                      #'487'
                     ]
    
    # === Adiciona features da janela a cada registro
    for erro in tipos_de_erros:
        df = df.withColumn(
            f"erro_{erro}_{horas}h", 
            F.sum(F.when(F.col("SIPSTATUSCODE_135") == erro, 1).otherwise(0)).over(window_janela)
        )
    
    df = df.withColumn(f"duracao_media_{horas}h", F.avg("ACCOUNTINGSESSIONTIME_13").over(window_janela))
    df = df.withColumn(f"origens_diferentes_{horas}h", F.countDistinct("IDENTIFICATIONNUMBER_A_10").over(window_janela))
    df = df.withColumn(f"portas_diferentes_{horas}h", F.countDistinct("NUMBER_A_IP_10").over(window_janela))
    df = df.withColumn(f"total_conexoes_{horas}h", F.count("*").over(window_janela))
    
    print(f'Features de janela de {horas}h adicionadas ao DataFrame.')
    
    return df


# === Função para múltiplas janelas de tempo
def adicionar_multiplas_janelas(df, lista_horas=[1, 6, 12, 24]):
    """
    Adiciona features para múltiplas janelas de tempo diferentes
    
    Args:
        df: DataFrame com os dados
        lista_horas: Lista com diferentes tamanhos de janela em horas
    
    Returns:
        DataFrame com features para todas as janelas especificadas
    """
    
    df_resultado = df
    
    for horas in lista_horas:
        print(f'Processando janela de {horas}h...')
        df_resultado = adicionar_features_janela_fixa(df_resultado, horas)
    
    print(f'Processamento concluído para janelas: {lista_horas}h')
    
    return df_resultado

### Dataset com Intervalo de Tempo

In [ ]:
# === Aplicar a função de agrupamento por origem e contagem de erro
df_engineered = agrupamentos_por_origem_janela_fixa(df_processed, 24)
df_engineered

In [ ]:
# === Salvar o DataFrame processado de volta ao S3
df_engineered.write.mode("overwrite").save_as_table("F_VOICETRAFFIC_SBC_ENGINEERED_404_24H")

### Dataset sem Intervalo de Tempo

In [ ]:
df_combined = df_processed.filter(
    (col("NUMBER_A_IP_10") == "177.154.149.216") &
    (col("SIPSTATUSCODE_135").isin(["200", "404", "487"]))
)

# Visualizar resultado
df_combined


In [ ]:
                       
df_filtred_404 = df_combined["DATAHORA",
                              "IDENTIFICATIONNUMBER_B_11",
                              "SIPSTATUSCODE_135"]

In [ ]:
df_filtred_404.write.mode("overwrite").save_as_table("F_VOICETRAFFIC_SBC_NOTIME")